<a href="https://colab.research.google.com/github/lhiwi/FUTURE_DS_02/blob/main/data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telco Customer Churn — 01: Data Audit & Cleaning
###  Goal: Produce a clean, analysis-ready dataset and a Power BI-ready export, with clear assumptions and quality checks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

path = "/content/drive/MyDrive/Future_DS/telco_churn.csv.csv"
df = pd.read_csv(path)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# Cell 4 — Schema & dtypes check (fundamental)

### df.info() gives a quick technical summary of the dataset:
### - number of rows (RangeIndex)
### - each column name
### - how many non-null values each column has (helps detect missing values)
### - the data type (dtype) of each column (helps detect wrong types)
### - memory usage (not critical now, but good to know)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


#  Duplicate checks (fundamental)

In [ ]:
# 1) Full-row duplicates:
# If >0, it means some rows are completely repeated.
dup_full_rows = df.duplicated().sum()

# 2) Duplicates by customerID:
# customerID should be unique per customer.
# If >0, the dataset may contain repeated customers, which affects churn rates and segment counts.
dup_customer_ids = df["customerID"].duplicated().sum()

print("Full-row duplicates:", dup_full_rows)
print("Duplicate customerID:", dup_customer_ids)

Full-row duplicates: 0
Duplicate customerID: 0


# Investigate TotalCharges cleanliness (fundamental)

In [ ]:
# 1) Show a few suspicious values: empty strings or whitespace-only
# We convert to string, strip spaces, and check if anything becomes empty.
empty_totalcharges = (df["TotalCharges"].astype(str).str.strip() == "").sum()

print("TotalCharges blank/whitespace entries:", empty_totalcharges)

# 2) Try converting TotalCharges to numeric safely:
# - errors="coerce" turns non-numeric values into NaN (this reveals hidden issues)
totalcharges_num = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("TotalCharges NaNs after numeric conversion:", totalcharges_num.isna().sum())

# 3) If there are NaNs, quickly look at the rows (tenure is usually 0)
if totalcharges_num.isna().sum() > 0:
    display(df.loc[totalcharges_num.isna(), ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]].head(10))

TotalCharges blank/whitespace entries: 11
TotalCharges NaNs after numeric conversion: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


#  Unique categories check (fundamental)

In [ ]:
key_cats = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod", "Churn"
]

for col in key_cats:
    print(f"\n{col} — unique values ({df[col].nunique()}):")
    print(df[col].value_counts(dropna=False))


gender — unique values (2):
gender
Male      3555
Female    3488
Name: count, dtype: int64

Partner — unique values (2):
Partner
No     3641
Yes    3402
Name: count, dtype: int64

Dependents — unique values (2):
Dependents
No     4933
Yes    2110
Name: count, dtype: int64

PhoneService — unique values (2):
PhoneService
Yes    6361
No      682
Name: count, dtype: int64

MultipleLines — unique values (3):
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

InternetService — unique values (3):
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

OnlineSecurity — unique values (3):
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

OnlineBackup — unique values (3):
OnlineBackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64

DeviceProtection

# Core cleaning (TotalCharges) + light feature prep for analysis/dashboard

In [ ]:
# 1) Convert TotalCharges to numeric.
#    - errors="coerce" turns blanks/invalid strings into NaN so we can handle them explicitly.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# 2) For customers with tenure == 0, total charges should logically be 0.
#    - We fill only those NaNs that are caused by tenure=0.
df.loc[(df["tenure"] == 0) & (df["TotalCharges"].isna()), "TotalCharges"] = 0

# 3) Quick verification: check if TotalCharges still has missing values.
print("TotalCharges NaNs after fix:", df["TotalCharges"].isna().sum())

# 4) Create a readable SeniorCitizen column (helps charts and dashboard filters)
df["SeniorCitizenFlag"] = np.where(df["SeniorCitizen"] == 1, "Yes", "No")

# 5) Create service flags for easier segmentation (optional but very helpful)
df["HasInternet"] = np.where(df["InternetService"] == "No", "No", "Yes")
df["HasPhone"] = np.where(df["PhoneService"] == "Yes", "Yes", "No")

# 6) Confirm dtypes of the new/updated columns
df[["TotalCharges", "SeniorCitizenFlag", "HasInternet", "HasPhone"]].dtypes

TotalCharges NaNs after fix: 0


,0
TotalCharges,float64
SeniorCitizenFlag,object
HasInternet,object
HasPhone,object


# Final data quality checks (fundamental before exporting)

In [ ]:
# 1) Confirm no missing values anywhere
missing_after = df.isna().sum().sort_values(ascending=False)
print("Top missing counts after cleaning:")
print(missing_after.head(10))

# 2) Confirm the dataframe still has the same number of rows/columns (sanity check)
print("\nShape:", df.shape)

# 3) Quick check: churn values are only Yes/No (no typos)
print("\nChurn unique values:", df["Churn"].unique())

# 4) Quick numeric sanity checks
print("\nTenure min/max:", df["tenure"].min(), df["tenure"].max())
print("MonthlyCharges min/max:", df["MonthlyCharges"].min(), df["MonthlyCharges"].max())
print("TotalCharges min/max:", df["TotalCharges"].min(), df["TotalCharges"].max())

Top missing counts after cleaning:
customerID         0
gender             0
SeniorCitizen      0
Partner            0
Dependents         0
tenure             0
PhoneService       0
MultipleLines      0
InternetService    0
OnlineSecurity     0
dtype: int64

Shape: (7043, 24)

Churn unique values: ['No' 'Yes']

Tenure min/max: 0 72
MonthlyCharges min/max: 18.25 118.75
TotalCharges min/max: 0.0 8684.8


#  Export a Power BI-ready cleaned dataset

In [ ]:
import os

# 1) Choose the folder you want to save into
out_dir = "/content/drive/MyDrive/Future_DS"

# 2) Make sure the folder exists
os.makedirs(out_dir, exist_ok=True)

# 3) Build the FULL file path (folder + filename)
output_path = os.path.join(out_dir, "telco_cleaned_for_powerbi.csv")

# 4) Save the file
df.to_csv(output_path, index=False)

print("Saved cleaned file to:", output_path)

Saved cleaned file to: /content/drive/MyDrive/Future_DS/telco_cleaned_for_powerbi.csv


In [ ]:
# Verify the file exists and show its size
import os
print("File exists:", os.path.exists(output_path))
print("File size (MB):", round(os.path.getsize(output_path) / (1024*1024), 2))

File exists: True
File size (MB): 1.0
